# Fix: Pinecone dimension mismatch (384 → 768)

**What happened:** swapping to `st-codesearch-distilroberta-base` changed embedding dim from 384 → 768.  
**Fix:** delete the old index, create a new one at 768, re-upsert everything.

In [ ]:
# ── Step 0: confirm dimensions before touching anything ───────────────────────
from sentence_transformers import SentenceTransformer

OLD_EMBEDDER = 'all-MiniLM-L6-v2'
NEW_EMBEDDER = 'flax-sentence-embeddings/st-codesearch-distilroberta-base'

test_text = "def hello(): return 'world'"

old_dim = len(SentenceTransformer(OLD_EMBEDDER).encode(test_text))
new_dim = len(SentenceTransformer(NEW_EMBEDDER).encode(test_text))

print(f"Old embedder ({OLD_EMBEDDER}): {old_dim} dims")
print(f"New embedder ({NEW_EMBEDDER}): {new_dim} dims")
print(f"\nIndex must be recreated at: {new_dim} dims")

In [ ]:
# ── Step 1: delete old index and create new one at correct dimension ──────────
from pinecone import Pinecone, ServerlessSpec

PINECONE_KEY = "YOUR_PINECONE_API_KEY"   # ← your key
INDEX_NAME   = "python-rag-v3"           # same name as in rag_pipeline.py
NEW_DIM      = new_dim                   # 768

pc = Pinecone(api_key=PINECONE_KEY)

# Check current state
existing = pc.list_indexes().names()
print(f"Existing indexes: {existing}")

if INDEX_NAME in existing:
    stats = pc.Index(INDEX_NAME).describe_index_stats()
    current_dim = stats.get('dimension', 'unknown')
    print(f"Current index dimension: {current_dim}")
    print(f"\nDeleting '{INDEX_NAME}'...")
    pc.delete_index(INDEX_NAME)
    print("Deleted.")

print(f"Creating '{INDEX_NAME}' at {NEW_DIM} dims...")
pc.create_index(
    name    = INDEX_NAME,
    dimension = NEW_DIM,
    metric  = "cosine",
    spec    = ServerlessSpec(cloud="aws", region="us-east-1")
)
print(f"✅ Index '{INDEX_NAME}' created at {NEW_DIM} dims")

In [ ]:
# ── Step 2: patch rag_pipeline.py to use the new embedder ─────────────────────
# Two things change in VersionControlRAG.__init__:
#   1. self.embedder = SentenceTransformer('flax-sentence-embeddings/st-codesearch-distilroberta-base')
#   2. index dimension = 768  (already handled above — index is now at 768)
#
# Also change ingest_data to embed code ONLY (drop Chinese summary):
#   OLD: vector_values = self.embedder.encode(summary + " " + code_content).tolist()
#   NEW: vector_values = self.embedder.encode(code_content).tolist()
#
# The patch below makes both changes in-place.

pipeline_path = "rag_pipeline.py"

with open(pipeline_path, 'r') as f:
    src = f.read()

# Patch 1: swap embedder
old_embed = "SentenceTransformer('all-MiniLM-L6-v2')"
new_embed = "SentenceTransformer('flax-sentence-embeddings/st-codesearch-distilroberta-base')"
assert old_embed in src, f"Could not find: {old_embed!r} — check rag_pipeline.py"
src = src.replace(old_embed, new_embed, 1)

# Patch 2: embed code-only (drop Chinese summary from vector)
old_vec = 'self.embedder.encode(summary + " " + code_content).tolist()'
new_vec = 'self.embedder.encode(code_content).tolist()  # code-only, no summary'
assert old_vec in src, f"Could not find embed line — check rag_pipeline.py"
src = src.replace(old_vec, new_vec, 1)

# Patch 3: update index dimension constant (create_index call)
old_dim_str = 'dimension=384'
new_dim_str = f'dimension={NEW_DIM}'
assert old_dim_str in src, f"Could not find dimension=384 — check rag_pipeline.py"
src = src.replace(old_dim_str, new_dim_str, 1)

with open(pipeline_path, 'w') as f:
    f.write(src)

print("✅ rag_pipeline.py patched:")
print(f"   embedder  → {new_embed}")
print(f"   ingest    → code-only embedding (no Chinese summary)")
print(f"   dimension → {NEW_DIM}")

In [ ]:
# ── Step 3: reload the patched pipeline and re-upsert corpus ──────────────────
import rag_pipeline, importlib, json, tqdm
from pathlib import Path

importlib.reload(rag_pipeline)
from rag_pipeline import VersionControlRAG

# Instantiate fresh — no adapter needed just for re-ingestion
rag = VersionControlRAG(
    pinecone_key = PINECONE_KEY,
    model_path   = "Qwen/Qwen2.5-Coder-7B",
    adapter_path = "Ivan17Ji/qwen-lora-250",
)

# Load corpus from disk
with open("local_corpus.json") as f:
    corpus_items = json.load(f)

print(f"Re-ingesting {len(corpus_items)} items with new {NEW_DIM}-dim embedder...")
print("This embeds code-only — no Chinese summaries in the vectors.")

rag._upsert_buffer = []
rag._corpus_ids    = set()

for item in tqdm.tqdm(corpus_items, desc="Upserting"):
    rag.ingest_data(
        doc_id       = item["id"],
        code_content = item["content"],
        min_version  = str(item["metadata"]["min_version"]),
        summary      = item["metadata"]["summary"],
    )

rag.flush_upsert_buffer()
rag._corpus_ids = {doc["id"] for doc in rag.local_corpus}

stats = rag.index.describe_index_stats()
print(f"\n✅ Done. Pinecone now has {stats.get('total_vector_count', 0)} vectors at {NEW_DIM} dims")

In [ ]:
# ── Step 4: restore BM25 and run the same diagnostic as before ────────────────
rag.load_local_corpus("local_corpus.json")

import re

def fp_strict(t): return re.sub(r'\s+', ' ', t).strip()[:80]
def fp_loose(t):  return re.sub(r'\s+', '', t)[:60]

# Load test records
with open("ragas_dataset.jsonl") as f:
    ragas_records = [json.loads(l) for l in f]
with open("fastapi_golden_set_splits.json") as f:
    splits = json.load(f)

test_questions = {x["instruction"] for x in splits["test"]}
test_records   = [r for r in ragas_records if r["question"] in test_questions]

# Run the same fingerprint diagnostic
r = test_records[0]
retrieved = rag.retrieve_complex(r["question"], target_version="3.10", top_k=3, mode="advanced")

print("QUESTION:", r["question"][:70])
print()
print("GOLDEN:")
for g in r["contexts"][:2]:
    print(f"  {fp_strict(g)!r}")
print()
print("RETRIEVED:")
golden_loose = {fp_loose(g) for g in r["contexts"]}
for ret in retrieved:
    hit = fp_loose(ret) in golden_loose
    icon = "HIT ✅" if hit else "miss ❌"
    print(f"  [{icon}] {fp_strict(ret)!r}")

In [ ]:
# ── Step 5: quick recall check across 10 test records ─────────────────────────
import random
random.seed(42)
sample = random.sample(test_records, min(10, len(test_records)))

hits, total = 0, 0
for rec in sample:
    ret = rag.retrieve_complex(rec["question"], target_version="3.10", top_k=3, mode="advanced")
    golden_loose = {fp_loose(g) for g in rec["contexts"]}
    hit = any(fp_loose(r) in golden_loose for r in ret)
    hits += int(hit)
    total += 1
    print(f"  {'✅' if hit else '❌'}  {rec['question'][:65]}")

print(f"\nHit rate@3 on {total} records: {hits/total:.0%}")
print("(Was ~0% before the fix — anything above 30% confirms the embedder was the problem)")